# 矩阵变换（Matrix Transformations）

对应课程：`phases/01-math-foundations/03-matrix-transformations`

> 矩阵是一台重塑空间的机器。搞清楚它对每个点做了什么，你就理解了整个变换。

本 notebook 把 `transformations.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `transformations.py`。

**贯穿全课的模式：** 矩阵的列就是新的基向量。先旋转再缩放，和先缩放再旋转，不是同一台机器。


## 0. 依赖

只用标准库。


In [1]:
import math


## 1. 作用到点：矩阵 × 向量、矩阵 × 矩阵

$$
(Av)_i = \sum_j A_{ij} v_j,\qquad
(AB)_{ij} = \sum_k A_{ik} B_{kj}
$$

左边的矩阵先作用。组合变换就是把两台机器乘在一起，再一次打到点上。


In [2]:
def mat_vec_mul(matrix, vector):
    """A @ v。每一行点一次向量。"""
    return [
        sum(matrix[i][j] * vector[j] for j in range(len(vector)))
        for i in range(len(matrix))
    ]


def mat_mul(a, b):
    """A @ B。左边先作用到右边的列上。"""
    rows_a, cols_b = len(a), len(b[0])
    cols_a = len(a[0])
    return [
        [sum(a[i][k] * b[k][j] for k in range(cols_a)) for j in range(cols_b)]
        for i in range(rows_a)
    ]


print("I @ [3, 4] =", mat_vec_mul([[1, 0], [0, 1]], [3, 4]))
print("[[2,0],[0,3]] @ [[1,1],[0,1]] =", mat_mul([[2, 0], [0, 3]], [[1, 1], [0, 1]]))


I @ [3, 4] = [3, 4]
[[2,0],[0,3]] @ [[1,1],[0,1]] = [[2, 2], [0, 3]]


## 2. 二维旋转

$$
R(\theta)=\begin{bmatrix}\cos\theta&-\sin\theta\\\sin\theta&\cos\theta\end{bmatrix}
$$

绕原点逆时针转 $\theta$。`[1,0]` 转 $90^\circ=\pi/2$ 应落到 `[0,1]`（浮点下接近）。


In [3]:
def rotation_2d(theta):
    """逆时针旋转矩阵。"""
    c, s = math.cos(theta), math.sin(theta)
    return [[c, -s], [s, c]]


p = [1.0, 0.0]
r90 = mat_vec_mul(rotation_2d(math.pi / 2), p)
print("R(90°) @ [1,0] =", [round(x, 10) for x in r90])


R(90°) @ [1,0] = [0.0, 1.0]


## 3. 缩放与剪切

$$
S=\begin{bmatrix}s_x&0\\0&s_y\end{bmatrix},\qquad
K=\begin{bmatrix}1&k_x\\k_y&1\end{bmatrix}
$$

缩放沿坐标轴拉长；剪切让矩形变成平行四边形（$k_x$ 把 $y$ 方向的位移加到 $x$）。


In [4]:
def scaling_2d(sx, sy):
    """沿 x / y 轴独立缩放。"""
    return [[sx, 0], [0, sy]]


def shearing_2d(kx, ky):
    """水平剪切 kx、竖直剪切 ky。"""
    return [[1, kx], [ky, 1]]


print("S(2,3) @ [1,1] =", mat_vec_mul(scaling_2d(2, 3), [1.0, 1.0]))
print("K(kx=1) @ [1,1] =", mat_vec_mul(shearing_2d(1, 0), [1.0, 1.0]))


S(2,3) @ [1,1] = [2.0, 3.0]
K(kx=1) @ [1,1] = [2.0, 1.0]


## 4. 反射

$$
R_x=\begin{bmatrix}1&0\\0&-1\end{bmatrix}
\quad\text{（对 x 轴镜像，y 变号）},\qquad
R_y=\begin{bmatrix}-1&0\\0&1\end{bmatrix}
\quad\text{（对 y 轴镜像，x 变号）}
$$

行列式为 $-1$：面积大小不变，但定向翻转。


In [5]:
def reflection_x():
    """对 x 轴反射。"""
    return [[1, 0], [0, -1]]


def reflection_y():
    """对 y 轴反射。"""
    return [[-1, 0], [0, 1]]


print("对 x 轴反射 [2,1] =", mat_vec_mul(reflection_x(), [2.0, 1.0]))
print("对 y 轴反射 [2,1] =", mat_vec_mul(reflection_y(), [2.0, 1.0]))


对 x 轴反射 [2,1] = [2.0, -1.0]
对 y 轴反射 [2,1] = [-2.0, 1.0]


## 5. 组合：顺序很重要

先旋转再缩放是 $S R$，先缩放再旋转是 $R S$。矩阵乘法不交换，两个结果一般不同。


In [6]:
R = rotation_2d(math.pi / 2)
S = scaling_2d(2, 0.5)
point = [1.0, 0.0]

rotate_then_scale = mat_mul(S, R)  # 先 R 后 S
scale_then_rotate = mat_mul(R, S)  # 先 S 后 R

print("点:", point)
print("先转 90° 再缩 (2, 0.5):", [round(x, 10) for x in mat_vec_mul(rotate_then_scale, point)])
print("先缩 (2, 0.5) 再转 90°:", [round(x, 10) for x in mat_vec_mul(scale_then_rotate, point)])
print("顺序不同，落点不同。")


点: [1.0, 0.0]
先转 90° 再缩 (2, 0.5): [0.0, 0.5]
先缩 (2, 0.5) 再转 90°: [0.0, 2.0]
顺序不同，落点不同。


## 6. $2\times 2$ 行列式：面积缩放

$$
\det\begin{bmatrix}a&b\\c&d\end{bmatrix}=ad-bc
$$

旋转保持面积和定向，$\det R(\theta)=\cos^2\theta+\sin^2\theta=1$。缩放的行列式是 $s_x s_y$。


In [7]:
def det_2x2(m):
    """2x2 行列式 = 平行四边形有向面积。"""
    return m[0][0] * m[1][1] - m[0][1] * m[1][0]


R = rotation_2d(math.pi / 4)
S = scaling_2d(2, 3)
print("det(R(45°)) =", round(det_2x2(R), 10), "  (应为 1)")
print("det(S(2,3)) =", det_2x2(S), "  (应为 6)")
print("det(reflect_y) =", det_2x2(reflection_y()), "  (翻转定向)")


det(R(45°)) = 1.0   (应为 1)
det(S(2,3)) = 6   (应为 6)
det(reflect_y) = -1   (翻转定向)


## 7. 特征值与特征向量

$$
A\mathbf{v}=\lambda\mathbf{v},\qquad
\lambda=\frac{\mathrm{tr}\pm\sqrt{\mathrm{tr}^2-4\det}}{2}
$$

特征向量是「只被拉长、不改变方向」的轴。对角缩放矩阵 $\mathrm{diag}(s_x,s_y)$ 的特征值就是 $s_x,s_y$，特征向量是坐标轴。


In [8]:
def eigenvalues_2x2(matrix):
    """特征方程 λ² - tr λ + det = 0。判别式为负则返回复数。"""
    a, b = matrix[0]
    c, d = matrix[1]
    trace = a + d
    det = a * d - b * c
    discriminant = trace ** 2 - 4 * det
    if discriminant < 0:
        real = trace / 2
        imag = (-discriminant) ** 0.5 / 2
        return (complex(real, imag), complex(real, -imag))
    sqrt_disc = discriminant ** 0.5
    return ((trace + sqrt_disc) / 2, (trace - sqrt_disc) / 2)


def eigenvector_2x2(matrix, eigenvalue):
    """(A - λI)v = 0 的一个单位解。"""
    a, b = matrix[0]
    c, d = matrix[1]
    if abs(b) > 1e-10:
        v = [b, eigenvalue - a]
    elif abs(c) > 1e-10:
        v = [eigenvalue - d, c]
    else:
        v = [1, 0] if abs(a - eigenvalue) < 1e-10 else [0, 1]
    mag = (v[0] ** 2 + v[1] ** 2) ** 0.5
    return [v[0] / mag, v[1] / mag]


S = [[3, 0], [0, 5]]
vals = eigenvalues_2x2(S)
print("diag(3,5) 特征值:", vals)
for lam in vals:
    vec = eigenvector_2x2(S, lam)
    Av = mat_vec_mul(S, vec)
    print(f"λ={lam}, v={vec}, A@v={Av}, λv={[lam * vec[0], lam * vec[1]]}")


diag(3,5) 特征值: (5.0, 3.0)
λ=5.0, v=[0.0, 1.0], A@v=[0.0, 5.0], λv=[0.0, 5.0]
λ=3.0, v=[1.0, 0.0], A@v=[3.0, 0.0], λv=[3.0, 0.0]


## 对照表

| 函数 | 角色 |
|------|------|
| `mat_vec_mul` / `mat_mul` | 把机器作用到点 / 把两台机器接起来 |
| `rotation_2d` | 绕原点转，面积不变 |
| `scaling_2d` | 沿轴拉长 |
| `shearing_2d` | 把矩形拧成平行四边形 |
| `reflection_x` / `reflection_y` | 镜像，定向翻转 |
| `det_2x2` | 有向面积缩放；旋转 ≈ 1 |
| `eigenvalues_2x2` | 沿不动方向的拉伸倍数 |
| `eigenvector_2x2` | 那个不动方向 |

要看完整打印 demo，运行：

```bash
python transformations.py
```
